# Pre-processamento do manual Volkswagen Polo 2025

Este notebook transforma o PDF do manual em Markdown, divide o conteúdo em trechos menores com metadados e ingere os embeddings em uma coleção Milvus para uso posterior em um fluxo RAG.

In [1]:
import logging
import time
from collections.abc import Iterable
from pathlib import Path
from dotenv import load_dotenv
import os

In [2]:
from docling_core.types.doc import ImageRefMode

In [3]:
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_milvus import Milvus
from sentence_transformers import SentenceTransformer
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from sentence_transformers import SentenceTransformer
from pymilvus import connections, utility, FieldSchema, CollectionSchema, DataType, Collection

## Configuração inicial

As células a seguir preparam o logger e definem qual formato de exportação do Docling será usado. Por padrão, o fluxo usa a exportação atual em Markdown (`USE_V2`) e mantém a exportação legada desativada.

In [4]:
_log = logging.getLogger(__name__)

In [5]:
USE_V2 = True
USE_LEGACY = False

## Exportação dos documentos convertidos

A função `export_documents` recebe os resultados da conversão do Docling, salva os documentos convertidos em Markdown e contabiliza conversões bem-sucedidas, parciais ou com falha.

In [6]:
def export_documents(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                conv_res.document.save_as_markdown(
                    output_dir / f"{doc_filename}.md",
                    image_mode=ImageRefMode.PLACEHOLDER,
                )

                # Exporta o documento Docling para Markdown:
                with (output_dir / f"{doc_filename}.md").open("w") as fp:
                    fp.write(conv_res.document.export_to_markdown())
                    
                _log.info(f"Saved: {doc_filename}.md")

            if USE_LEGACY:
                
                # Exporta no formato Markdown:
                with (output_dir / f"{doc_filename}.legacy.md").open(
                    "w", encoding="utf-8"
                ) as fp:
                    fp.write(conv_res.legacy_document.export_to_markdown())

                _log.info(f"Saved: {doc_filename}.md")
                
        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            _log.info(
                f"Document {conv_res.input.file} was partially converted with the following errors:"
            )
            for item in conv_res.errors:
                _log.info(f"\t{item.error_message}")
            partial_success_count += 1
        else:
            _log.info(f"Document {conv_res.input.file} failed to convert.")
            failure_count += 1

    _log.info(
        f"Processed {success_count + partial_success_count + failure_count} docs, "
        f"of which {failure_count} failed "
        f"and {partial_success_count} were partially converted."
    )
    return success_count, partial_success_count, failure_count

## Conversão do PDF para Markdown

A função `main` configura o pipeline de PDF do Docling, aponta para o arquivo `Volkswagen_Polo_2025.pdf` e grava o resultado em `markdown_manuals/`. Essa etapa cria a versão textual do manual que será usada na ingestão.

In [7]:
def main():
    logging.basicConfig(level=logging.INFO)

    input_doc_paths = [
        Path("./Volkswagen_Polo_2025.pdf")
    ]

    pipeline_options = PdfPipelineOptions()
    pipeline_options.generate_page_images = False

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=DoclingParseV4DocumentBackend
            )
        }
    )

    start_time = time.time()

    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,  # permite concluir todas as conversões e avaliar os resultados no fim
    )
    success_count, partial_success_count, failure_count = export_documents(
        conv_results, output_dir=Path("markdown_manuals")
    )

    end_time = time.time() - start_time

    _log.info(f"Document conversion complete in {end_time:.2f} seconds.")

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )

## Execução da conversão

Esta célula executa o fluxo de conversão quando o notebook ou script é rodado diretamente.

In [8]:
if __name__ == "__main__":
    main()

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
/Users/thaismedeiros/miniconda3/envs/reasoning/lib/python3.13/site-packages/docling/datamodel/document.py:221: FutureWarning: DoclingParseV4DocumentBackend was removed in docling 2.74.0 and will raise an error in a future release. Use DoclingParseDocumentBackend instead.
  self._backend = backend(self, path_or_stream=path_or_stream)
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash da8ee54367e24a411bb95591847bb457
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
INFO

## Seleção dos arquivos Markdown

Depois da conversão, esta lista define quais arquivos Markdown serão processados para extração de chunks e criação dos embeddings.

In [9]:
markdown_paths = [
    "./markdown_manuals/Volkswagen_Polo_2025.md",
]

## Chunking, metadados e ingestão no Milvus

Este bloco carrega cada Markdown, extrai marca, modelo e ano a partir do nome do arquivo, divide o texto primeiro por cabeçalhos e depois por tamanho quando necessário. Em seguida, gera embeddings normalizados com `all-MiniLM-L6-v2`, cria a coleção `manuals_open` no Milvus se ela ainda não existir e insere os chunks com seus metadados.

In [10]:
all_documents = []

for markdown_path in markdown_paths:
    _log.info(f"--- Processando arquivo: {markdown_path} ---")

    if not os.path.exists(markdown_path):
        _log.warning(f"Arquivo não encontrado, pulando: {markdown_path}")
        continue

    filename = Path(markdown_path).stem
    try:
        brand, model, year = filename.split('_')
        _log.info(f"Metadados extraídos: Marca={brand}, Modelo={model}, Ano={year}")
    except ValueError:
        _log.error(f"O nome do arquivo '{filename}.md' não segue o padrão 'marca_modelo_ano'. Pulando.")
        continue

    loader = UnstructuredMarkdownLoader(markdown_path, mode="single")
    docs = loader.load()

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
        strip_headers=False
    )
    header_chunks = header_splitter.split_text(docs[0].page_content)

    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    final_chunks_for_file = []
    for chunk in header_chunks:
        chunk.metadata['brand'] = brand
        chunk.metadata['model'] = model
        chunk.metadata['year'] = year
        chunk.metadata['source'] = filename

        if len(chunk.page_content) > 1000:
            final_chunks_for_file.extend(char_splitter.split_documents([chunk]))
        else:
            final_chunks_for_file.append(chunk)
    
    all_documents.extend(final_chunks_for_file)
    _log.info(f"Arquivo processado. {len(final_chunks_for_file)} chunks foram criados e adicionados.")

if not all_documents:
    _log.warning("Nenhum documento para ingerir. Finalizando o script.")
else:
    _log.info(f"Total de {len(all_documents)} chunks de todos os arquivos prontos para ingestão.")

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    texts = [doc.page_content for doc in all_documents]
    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    MILVUS_HOST = "localhost"
    MILVUS_PORT = "19531"
    MILVUS_COLLECTION_NAME = "manuals_open"

    connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)

    if utility.has_collection(MILVUS_COLLECTION_NAME):
        collection = Collection(MILVUS_COLLECTION_NAME)
    else:
        fields = [
            FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
            FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
            FieldSchema(name="brand", dtype=DataType.VARCHAR, max_length=100),
            FieldSchema(name="model", dtype=DataType.VARCHAR, max_length=100),
            FieldSchema(name="year", dtype=DataType.VARCHAR, max_length=20),
            FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=255),
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=embeddings.shape[1]),
        ]
        schema = CollectionSchema(fields, description="Manual chunks")
        collection = Collection(name=MILVUS_COLLECTION_NAME, schema=schema)

        index_params = {
            "index_type": "IVF_FLAT",
            "metric_type": "IP",
            "params": {"nlist": 1024}
        }
        collection.create_index(field_name="embedding", index_params=index_params)

    data = [
        texts,
        [doc.metadata["brand"] for doc in all_documents],
        [doc.metadata["model"] for doc in all_documents],
        [doc.metadata["year"] for doc in all_documents],
        [doc.metadata["source"] for doc in all_documents],
        embeddings.tolist()
    ]

    collection.insert(data)
    collection.load()

    _log.info("Todos os documentos foram ingeridos no Milvus com sucesso!")

INFO:__main__:--- Processando arquivo: ./markdown_manuals/Volkswagen_Polo_2025.md ---
INFO:__main__:Metadados extraídos: Marca=Volkswagen, Modelo=Polo, Ano=2025
INFO:__main__:Arquivo processado. 1268 chunks foram criados e adicionados.
INFO:__main__:Total de 1268 chunks de todos os arquivos prontos para ingestão.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Batches: 100%|██████████| 40/40 [00:16<00:00,  2.43it/s]
INFO:__main__:Todos os documentos foram ingeridos no Milvus com sucesso!


## Conexão LangChain com a coleção vetorial

Por fim, o notebook instancia um `Milvus` vector store apontando para a coleção `manuals_open`, permitindo reutilizar os vetores ingeridos em etapas posteriores de busca semântica ou RAG.

In [11]:
vectorstore = Milvus(
    collection_name="manuals_open",
    embedding_function=model,
    connection_args={"uri": "http://localhost:19531"},
    text_field="text",
    vector_field="embedding",
    auto_id=True,
    consistency_level="Strong",
    search_params={"metric_type": "IP", "params": {"nprobe": 10}},
    drop_old=False
)